# Brain Walk · Notebook 3 : les rythmes de l'EEG

PSY2007D — Laboratoire 1, automne 2026

<table style="width:100%; border-collapse:separate; border-spacing:0; margin:6px 0 10px 0;"><tr><td style="background:#EEF3F9; color:#0B2545; padding:10px 14px; border:3px solid #FFFFFF; border-radius:8px; vertical-align:top; width:25%;"><div style="font-size:0.8em; color:#5B6472;">NOTEBOOK 1</div><div style="font-weight:bold; font-size:1.05em; color:#0B2545;">Prétraitement</div><div style="font-size:0.85em; color:#5B6472;">lire · nettoyer · époques</div></td><td style="background:#EEF3F9; color:#0B2545; padding:10px 14px; border:3px solid #FFFFFF; border-radius:8px; vertical-align:top; width:25%;"><div style="font-size:0.8em; color:#5B6472;">NOTEBOOK 2</div><div style="font-weight:bold; font-size:1.05em; color:#0B2545;">ERP</div><div style="font-size:0.85em; color:#5B6472;">P2 · N400 · LPC · hypothèses H1-H10</div></td><td style="background:#0B2545; color:#FFFFFF; padding:10px 14px; border:3px solid #FFFFFF; border-radius:8px; vertical-align:top; width:25%;"><div style="font-size:0.8em; color:#C9D6E8;">NOTEBOOK 3</div><div style="font-weight:bold; font-size:1.05em; color:#FFFFFF;">Rythmes</div><div style="font-size:0.85em; color:#C9D6E8;">spectres · temps-fréquence</div></td><td style="background:#EEF3F9; color:#0B2545; padding:10px 14px; border:3px solid #FFFFFF; border-radius:8px; vertical-align:top; width:25%;"><div style="font-size:0.8em; color:#5B6472;">NOTEBOOK 4</div><div style="font-weight:bold; font-size:1.05em; color:#0B2545;">Apprentissage</div><div style="font-size:0.85em; color:#5B6472;">décoder · ERP et rythmes</div></td></tr></table>

Le notebook 2 mesure les ERP, la moyenne du signal **au même instant** après le mot. Une partie de l'activité n'y apparaît pas : les
**rythmes**, des oscillations du signal (en Hz, le nombre de cycles par seconde). Leur **puissance** (leur amplitude au carré, en µV²)
peut monter ou baisser après le mot sans que leur **phase** (le moment du cycle) soit calée sur lui : dans la moyenne des essais, ces
oscillations s'annulent. Ce notebook mesure cette puissance, fréquence par fréquence, pour les mêmes 17 participants, les mêmes
conditions et les mêmes hypothèses H1 à H10.

Chaque mesure est un cadre (schéma) : une **bande** (un intervalle de fréquences : thêta 4-7 Hz, alpha 8-12 Hz, bêta 13-30 Hz) × une
**fenêtre** (un intervalle de temps après le mot, ou avant la **tape**, le toucher du participant sur la tablette pour répondre) × une
**région** (les électrodes moyennées ensemble).

![mesures](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_3_rythmes/mesures_tf.png)

| Partie | Contenu |
|---|---|
| 0 | Les données, **votre hypothèse**, les options |
| 1 | Les spectres, avec et sans la tâche ; la pente et la bosse |
| 2 | La puissance au cours du temps (temps-fréquence) ; la cohérence de phase |
| 3 | Les topographies |
| 4 | Une mesure par participant : **le test de votre hypothèse** |
| 5 | L'interaction avec la posture (H4, H8, H10) |
| 6 | L'EEG et le comportement (H5, H9) |
| 7 | À retenir pour votre rapport, et des pistes pour aller plus loin |
| 8 | **Défis (points bonus)** |

<div style="color:#1F2937; background:#F9FAFB; border:1px solid #E5E7EB; border-radius:6px; padding:8px 14px;"><b>Trois types de blocs</b><div style="margin:3px 0;"><span style="background:#0B5CAD; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 1</span> &nbsp;le code est complet : exécutez la cellule et lisez les commentaires.</div><div style="margin:3px 0;"><span style="background:#E8710A; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 2</span> &nbsp;complétez une ou deux lignes marquées <code>&lt;---</code>.</div><div style="margin:3px 0;"><span style="background:#7C3AED; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 3</span> &nbsp;écrivez la cellule vous-même, à partir des indices et des liens vers la documentation.</div><div style="font-size:0.9em; color:#5B6472; margin-top:4px;">Chaque exercice est suivi d'une solution repliée : essayez d'abord.</div></div>

<details><summary>Sur votre ordinateur ou dans Colab ?</summary>

- **Sur votre ordinateur (recommandé)** : les données sont déjà dans `tasks/brainwalk/` si vous avez fait le notebook 1.
- **Colab** : la première cellule installe MNE et specparam et télécharge les données (2,2 Go) à **chaque** session.
- Le temps-fréquence (partie 2) prend environ 1 minute.
</details>

In [ ]:
# ---- 0.1 Les paquets (installés s'il en manque), les imports et les données
import importlib.util
import sys
import warnings

DANS_COLAB = "google.colab" in sys.modules
DEPOT = "https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/"
if DANS_COLAB:                            # Colab n'a pas les fichiers du cours : on les télécharge
    !wget -q -N {DEPOT}requirements.txt {DEPOT}outils_brainwalk.py
PAQUETS = ["mne", "specparam"]   # les paquets de ce notebook
if any(importlib.util.find_spec(p) is None for p in PAQUETS):   # il en manque : installer ceux du cours (une fois)
    %pip install -q -r requirements.txt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from scipy import stats
import outils_brainwalk as ob             # fonctions du projet (fichier outils_brainwalk.py, à côté du notebook)

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore", message="Mean of empty slice")   # une électrode sans mesure dans une moyenne : NaN, sans message
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
DOSSIER = ob.telecharger_donnees()        # déjà là si vous avez fait le notebook 1
print("MNE", mne.__version__)

## 0. Les données, votre hypothèse et les options

- Les **époques** du notebook 2 (le segment d'EEG découpé autour de chaque mot, ou de chaque tape), ici **plus longues** (−1,0 à 1,5 s
  autour du mot, −1,5 à 1,0 s autour de la tape), à 100 Hz : pour mesurer la puissance à 4 Hz à un instant, il faut plusieurs centaines
  de millisecondes de signal autour (partie 2).
- Les enregistrements **sans tâche** (repos assis, marche seule), coupés en **segments** de 2 s (des morceaux successifs de l'enregistrement ;
  la « fenêtre » reste le temps de votre mesure) : la **référence** de chaque posture (partie 1).
- Comme au notebook 2, la colonne `mauvaises` liste les électrodes **non mesurées** dans chaque époque : une région = la moyenne de ses
  seules électrodes mesurées.

In [ ]:
# ---- 0.2 Charger les époques longues, les enregistrements sans tâche et le comportement
dossier = DOSSIER / 'pretraite_00'
epochs, epochs_tape = {}, {}                                 # participant -> ses époques longues autour du mot, et autour de la tape
for fichier in sorted((dossier / 'longues').glob('sub-*_longues_epo.fif')):
    participant = fichier.name.split('_')[0][4:]             # 'sub-P003_longues_epo.fif' -> 'P003'
    epochs[participant] = mne.read_epochs(fichier, preload=True)
    epochs_tape[participant] = mne.read_epochs(dossier / 'longues' / 'tape' / f'sub-{participant}_tape_longues_epo.fif', preload=True)
if not epochs:
    raise ValueError('Les époques longues (pretraite_00/longues/) sont introuvables : relancez la cellule 0.1, elle télécharge la '
                     'dernière version des données')
enregistrements = pd.read_csv(dossier / 'sans_tache' / 'enregistrements.tsv', sep='\t', keep_default_na=False)
sans_tache = {(e.participant, e.enregistrement): mne.read_epochs(dossier / 'sans_tache' / f'sub-{e.participant}_{e.enregistrement}_epo.fif',
                                                                 preload=True) for e in enregistrements.itertuples()}   # -> segments de 2 s
essais = pd.read_csv(DOSSIER / 'comportement' / 'essais.csv')
noms_ch = next(iter(epochs.values())).ch_names               # les 19 électrodes, dans l'ordre des fichiers
print(f'{len(epochs)} participants, {len(sans_tache)} enregistrements sans tâche, {len(essais)} essais de comportement')
resume = pd.DataFrame({p: ep.metadata['posture'].value_counts() for p, ep in epochs.items()}).T.fillna(0).astype(int).add_prefix('époques ')
resume = resume.join(enregistrements.pivot(index='participant', columns='enregistrement', values='fenetres_gardees').astype('Int64')
                     .add_prefix('segments '))
resume.join(enregistrements.groupby('participant')['ordre'].first())   # époques, segments de 2 s gardés et ordre de la séance

### Les hypothèses H1 à H10, en rythmes

Les mêmes hypothèses qu'au notebook 2. Chacune y portait sur une composante de l'ERP ; ici, sur la puissance d'un rythme : une
**bande** × une **fenêtre** × une **région**. Les fenêtres des ERP (180-250 ms pour la P200) sont trop courtes pour les rythmes lents : à
4-7 Hz, la puissance à un instant résume déjà 250 à 330 ms de signal.

![les hypothèses en rythmes](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_3_rythmes/hypotheses_rythmes.png)

| | Hypothèse (notebook 2) | A / B | Mesure | Bande | Fenêtre | Parties |
|---|---|---|---|---|---|---|
| H1 | La marche modifie la P200 | assis / marche | thêta précoce ; refaites avec l'alpha | 4-7 Hz | 100-300 ms | 1 à 4 |
| H2 | LF diffère de HF dès 180-300 ms | HF / LF | thêta précoce | 4-7 Hz | 100-300 ms | 1 à 4 |
| H3 | La N400 est plus négative pour LF | HF / LF | thêta, avec la cohérence de phase (2.3) | 4-7 Hz | 300-500 ms | 1 à 4 |
| H4 | L'effet de fréquence sur la N400 change en marche | HF / LF, assis puis en marche | thêta | 4-7 Hz | 300-500 ms | 1 à 4, puis **5** |
| H5 | L'effet N400 suit l'effet sur le comportement | HF / LF | thêta | 4-7 Hz | 300-500 ms | 1 à 4, puis **6** |
| H6 | La marche modifie la P3/LPC | assis / marche | thêta tardif ; refaites avec l'alpha | 4-7 Hz | 500-800 ms | 1 à 4 |
| H7 | La P3/LPC diffère entre LF et HF | HF / LF | thêta tardif | 4-7 Hz | 500-800 ms | 1 à 4 |
| H8 | L'effet de fréquence sur la P3/LPC change en marche | HF / LF, assis puis en marche | thêta tardif | 4-7 Hz | 500-800 ms | 1 à 4, puis **5** |
| H9 | Avant une réponse lente, la CPP monte autrement | rapides / lents | bêta (tape) | 13-30 Hz | 500 ms avant la tape | 1 à 4, puis **6** |
| H10 | La marche modifie la CPP, et son effet de fréquence | assis / marche | bêta (tape) | 13-30 Hz | 500 ms avant la tape | 1 à 4, puis **5** |

![feuille de route](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_3_rythmes/feuille_de_route.png)

<div style="border-left:5px solid #4A3AA7; background:#F1EFFB; color:#1F2937; padding:10px 16px; border-radius:6px; margin:8px 0;"><b style="color:#4A3AA7;">Ce que dit la littérature</b><br><ul style='margin:4px 0 0 0;'><li><b>Thêta</b> (4-7 Hz) : monte de 300 à 500 ms après un mot, avec la récupération lexicale (Bastiaansen et al., 2005) ; le thêta précoce est surtout la P2 vue en fréquences (Klimesch et al., 2004).</li><li><b>Alpha</b> (8-12 Hz) : baisse pendant la lecture ; une baisse liée à une tâche est plus petite en marche (Protzak et Gramann, 2021).</li><li><b>Bêta</b> (13-30 Hz) : baisse avant le mouvement, jusqu'à la réponse (Steinemann et al., 2018).</li><li>En marche, le bruit des pas est surtout <b>sous 8,5 Hz</b> (Gwin et al., 2010), dans le thêta : pour la posture, refaites avec l'alpha.</li></ul></div>

<div style="border-left:5px solid #E8710A; background:#FDF1E7; color:#1F2937; padding:10px 16px; border-radius:6px; margin:8px 0;"><b style="color:#E8710A;">À vérifier</b><br>L'énoncé officiel des hypothèses est celui du cours (voir le notebook 2). La mesure de
chaque hypothèse est un choix de ce notebook (Keil et al., 2022) : justifiez-le dans votre rapport.</div>

### <span style="background:#E8710A; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 2</span> 0.3 Votre hypothèse

**Une ligne à changer** : `H = 'H3'`. Le tableau de la cellule est celui du dessus, écrit en Python : pour chaque hypothèse, ce qu'on
compare, la mesure, sa bande (en Hz), sa fenêtre (en secondes) et sa région. La cellule en tire les cinq réglages du notebook.

In [ ]:
# ---- 0.3 Votre hypothèse
H = 'H3'                       # 'H1' à 'H10' : la seule ligne à changer

FC, CP, PO, C = ob.REGION_FC, ob.REGION_CP, ob.REGION_PO, ob.REGION_C   # les régions : F3 à C4, C3 à P4, P3 à O2, et C3 Cz C4
HYPOTHESES = {                 # H : (ce qu'on compare, la mesure, la bande en Hz, la fenêtre en s, la région) : le tableau du dessus
    'H1': ('posture', 'thêta précoce', (4, 7), (0.1, 0.3), FC),
    'H2': ('frequence', 'thêta précoce', (4, 7), (0.1, 0.3), FC),
    'H3': ('frequence', 'thêta', (4, 7), (0.3, 0.5), CP),
    'H4': ('frequence', 'thêta', (4, 7), (0.3, 0.5), CP),            # puis la partie 5
    'H5': ('frequence', 'thêta', (4, 7), (0.3, 0.5), CP),            # puis la partie 6
    'H6': ('posture', 'thêta tardif', (4, 7), (0.5, 0.8), CP),
    'H7': ('frequence', 'thêta tardif', (4, 7), (0.5, 0.8), CP),
    'H8': ('frequence', 'thêta tardif', (4, 7), (0.5, 0.8), CP),     # puis la partie 5
    'H9': ('tr', 'bêta (tape)', (13, 30), (-0.5, 0.0), C),           # autour de la tape ; puis la partie 6
    'H10': ('posture', 'bêta (tape)', (13, 30), (-0.5, 0.0), C),     # autour de la tape ; puis la partie 5
}
HYPOTHESE, MESURE, BANDE, FENETRE, REGION = HYPOTHESES[H]
print(f"{H} : on compare « {HYPOTHESE} » sur le {MESURE}, {BANDE[0]}-{BANDE[1]} Hz, de {FENETRE[0] * 1000:.0f} à "
      f"{FENETRE[1] * 1000:.0f} ms, région {', '.join(REGION)}")

### <span style="background:#E8710A; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 2</span> 0.4 Les options (facultatif)

Pour tester votre hypothèse telle quelle, **laissez cette cellule comme elle est**. Chaque option répond à une question qu'on peut se
poser ensuite ; changez-en une, puis relancez les cellules à partir de 0.5 :

| Option | La question | Où regarder |
|---|---|---|
| `POSTURE` | L'effet existe-t-il dans une seule posture ? | parties 2 à 4 |
| la ligne de l'**alpha** | Pour une hypothèse de posture : l'effet tient-il hors du thêta, là où les pas font moins de bruit ? | parties 1 à 4 |
| `N_CYCLES` | Le compromis entre la précision en temps et en fréquence change-t-il l'effet ? | 2.4, puis partie 4 |
| `BANDE`, `FENETRE`, `REGION` | Une autre mesure ? **À décider avant de voir les résultats** | partout |

In [ ]:
# ---- 0.4 Les options : laissez-les telles quelles pour tester votre hypothèse
POSTURE = 'les deux'           # pour 'frequence' et 'tr' : 'les deux', 'assis' ou 'marche'
PARTICIPANT = 'P010'           # le participant des essais un par un (2.2)
N_CYCLES = ob.N_CYCLES_TF      # les cycles des ondelettes (partie 2) : de 3 cycles à 3 Hz à 10 cycles à 30 Hz
# MESURE, BANDE, FENETRE, REGION = 'alpha', (8, 12), (0.3, 0.8), PO   # l'alpha (posture) : enlevez le # au début de la ligne
# BANDE = (8, 10)              # une autre bande (Hz)
# FENETRE = (0.40, 0.70)       # une autre fenêtre (s)
# REGION = ['Cz']              # d'autres électrodes

### 0.5 Préparer les conditions

La cellule suivante choisit le **zéro des temps** (le mot, ou la tape pour une mesure « (tape) »), les deux conditions comparées, A et B,
avec leurs postures et leurs types de mots, et, pour H9, coupe les essais en **rapides** et **lents** (de part et d'autre du TR médian du
participant, dans sa posture et son type de mot).

In [ ]:
# ---- 0.5 Préparer les conditions selon vos choix (le code : outils_brainwalk.py)
CENTRE = 'tape' if '(tape)' in MESURE else 'mot'            # le zéro des temps
ZERO = 'le mot' if CENTRE == 'mot' else 'la tape'
COND_A, COND_B, CONDITIONS = ob.conditions_tf(HYPOTHESE, POSTURE)   # A et B (l'effet = B - A), avec leurs postures et types de mots
COULEURS = {COND_A: '#2f6db5', COND_B: '#e0791d'}
EPOQUES = epochs_tape if CENTRE == 'tape' else epochs
if HYPOTHESE == 'tr':
    ob.ajouter_vitesse(epochs, epochs_tape)                  # la colonne vitesse : 'rapides' ou 'lents'
j = [noms_ch.index(c) for c in REGION]                       # votre région, en numéros d'électrodes
NOM = MESURE if ob.MESURES_TF.get(MESURE) == (BANDE, FENETRE, REGION) else 'votre mesure'   # le nom affiché dans les figures
PARTIES = '1 à 4' + {'H4': ', puis 5', 'H8': ', puis 5', 'H10': ', puis 5', 'H5': ', puis 6', 'H9': ', puis 6'}.get(H, '')
print(f'{H} (parties {PARTIES}) : {COND_B} comparé à {COND_A} ; {NOM} : {BANDE[0]} à {BANDE[1]} Hz, de {FENETRE[0] * 1000:.0f} à '
      f'{FENETRE[1] * 1000:.0f} ms (0 : {ZERO}), région {", ".join(REGION)}')

## 1. Les spectres, avec et sans la tâche

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="color:#0B5CAD; font-weight:bold;">toutes les hypothèses</span></div>

Avant de suivre la puissance dans le temps (partie 2), on regarde sa moyenne, fréquence par fréquence.

![spectre](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_3_rythmes/spectre.png)

- **Spectre** : la puissance de chaque fréquence, moyennée sur le temps (en µV²/Hz).
- **Welch** : couper le signal en segments de 2 s, calculer le spectre de chacune, puis en faire la moyenne (moins bruité) ; un point
  tous les 0,5 Hz.
- **dB (décibel)** : 10 × log10 d'une puissance, ou du rapport de deux puissances. 0 dB : égales ; +3 dB : deux fois plus ; +10 dB : dix
  fois plus ; −3 dB : deux fois moins.

Quatre situations par participant : le **repos** assis, la **tâche assis** (−0,5 à 1,5 s autour de chaque mot), la **marche seule** et la
**tâche en marche**. Comparer la tâche à la même posture **sans** la tâche (la **référence** : le repos pour assis, la marche seule pour la
marche) sépare ce que la tâche change de ce que la marche ajoute (cerveau **et** bruit des pas). Limite : la marche seule vient toujours
avant les blocs en marche, et une tâche change la marche elle-même (Malcolm et al., 2018).

In [ ]:
# ---- 1.1 Le spectre de chaque participant dans les quatre situations (le code : ob.spectres_situations)
spectres, freqs_psd, quatre = ob.spectres_situations(epochs, sans_tache, enregistrements)   # (participant, situation) -> spectre
manque = {p: [s for s in ob.SITUATIONS if (p, s) not in spectres] for p in epochs if p not in quatre}
print(f'{len(quatre)} participants ont les quatre situations ; il manque : '
      + ('; '.join(f'{p} ({", ".join(s)})' for p, s in manque.items()) or 'rien'))

In [ ]:
# ---- F1 : le spectre des quatre situations, dans trois régions (le code de la figure : ob.figure_spectres)
ob.figure_spectres(spectres, quatre, freqs_psd, noms_ch)

**Lire.** Quelles fréquences la marche seule élève-t-elle par rapport au repos ? Le pic de l'alpha, vers 10 Hz, reste-t-il visible en
marche ?

**1.2** : votre bande dans votre région, dans les quatre situations. Chaque différence est testée contre 0 (test t ; **dz** = la
différence moyenne / son écart-type, comme au notebook 2), en dB puis en µV²/Hz. L'**interaction** : ce que la tâche change en marche,
moins ce qu'elle change assis.

In [ ]:
# ---- 1.2 Votre bande dans les quatre situations, et les comparaisons (le code : ob.comparer_situations)
dans_bande_psd = (freqs_psd >= BANDE[0]) & (freqs_psd <= BANDE[1])
puissance_4 = ob.puissance_situations(spectres, quatre, j, dans_bande_psd)   # participants x situations (µV²/Hz)
print(f'{BANDE[0]} à {BANDE[1]} Hz, région {", ".join(REGION)}, {len(puissance_4)} participants')
pd.concat({'dB': ob.comparer_situations(10 * np.log10(puissance_4)), 'µV²/Hz': ob.comparer_situations(puissance_4)})

In [ ]:
# ---- F2 : votre bande avec et sans la tâche, dans chaque posture (ob.figure_bande_tache)
ob.figure_bande_tache(10 * np.log10(puissance_4), BANDE, REGION)

### <span style="background:#E8710A; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 2</span> À vous de compléter : un participant

Avec le dictionnaire `spectres` (clé : `(participant, situation)`, valeur : électrodes × fréquences en µV²/Hz), calculez pour P010 la
puissance de votre bande dans votre région, en dB, au **repos** et en **marche seule**.

In [ ]:
# ---- Exercice
# for situation in ["repos", "marche seule"]:
#     s = spectres[("P010", ...)][j][:, dans_bande_psd]          # <--- à compléter : j (votre région, 0.5), dans_bande_psd (1.2)
#     print(situation, round(10 * np.log10(np.nanmean(s)), 2), "dB")

In [ ]:
#@title Solution (à consulter après avoir essayé)
for situation in ["repos", "marche seule"]:
    s = spectres[("P010", situation)][j][:, dans_bande_psd]
    print(situation, round(10 * np.log10(np.nanmean(s)), 2), "dB")
# En marche, les électrodes sèches bougent à chaque pas : la puissance monte surtout aux basses fréquences.

### 1.3 Le rythme au-dessus de la pente

Un spectre d'EEG descend toujours avec la fréquence : c'est la **pente** (l'activité « 1/f », ou apériodique, qui n'est pas un rythme ;
une pente plus grande = un spectre plus raide). Un rythme forme une **bosse** au-dessus de cette pente. specparam (anciennement FOOOF ;
séance 5) ajuste la droite et mesure ce qui la dépasse dans votre bande : « au-dessus de la pente (dB) ». R² : la qualité de
l'ajustement (1 = parfait).

![pente et bosse](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_3_rythmes/pente_bosse.png)

In [ ]:
# ---- 1.3 La pente et la bosse dans chaque situation (le code : ob.pente_et_bosse ; Donoghue et al., 2020)
pente_bosse = ob.pente_et_bosse(spectres, puissance_4.index, freqs_psd, j, dans_bande_psd)   # une ligne par participant et situation
tests = {mesure: ob.comparer_situations(pente_bosse.pivot(index='participant', columns='situation', values=mesure))
         for mesure in ('pente', 'au-dessus de la pente (dB)')}
display(pente_bosse.groupby('situation')[['pente', 'au-dessus de la pente (dB)', 'pic alpha (Hz)', 'R²']].mean().loc[ob.SITUATIONS].round(2))
pd.concat(tests)                                             # les comparaisons, pour la pente puis pour la bosse

**Lire.** La marche seule change-t-elle surtout la **pente** ou la **bosse** de votre bande ? Si c'est la pente, les comparaisons en dB
de 1.2 en dépendent (Gyurkovics et al., 2021).

### <span style="background:#5B6472; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Pour aller plus loin</span> 1.4 Le repos du début et le repos de la fin

Le repos assis est enregistré au début et à la fin de la séance (environ une heure d'écart, la tâche entre les deux). Votre bande
change-t-elle entre les deux ? Cela compte : la référence assis est le repos le plus proche des blocs assis.

In [ ]:
# ---- 1.4 Le repos de la fin moins le repos du début (le code : ob.repos_debut_fin)
repos_2 = ob.repos_debut_fin(sans_tache, enregistrements, epochs, noms_ch, REGION, dans_bande_psd)
t, p = stats.ttest_1samp(repos_2['fin - début (dB)'], 0.0)
print(f'{BANDE[0]}-{BANDE[1]} Hz, {", ".join(REGION)} : fin - début {repos_2["fin - début (dB)"].mean():+.2f} dB, t({len(repos_2) - 1}) = {t:.2f}, '
      f'p = {p:.3f} ; selon l\'ordre de la séance : {repos_2.groupby("ordre")["fin - début (dB)"].mean().round(2).to_dict()}')
repos_2.round(2)

## 2. La puissance au cours du temps

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="color:#0B5CAD; font-weight:bold;">toutes les hypothèses</span></div>

Le spectre (partie 1) moyenne tout l'essai. Le **temps-fréquence** montre **quand** la puissance change : une carte avec le temps en
abscisse, la fréquence en ordonnée et la puissance en couleur.

![Morlet](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_3_rythmes/morlet.png)

- **Ondelette de Morlet** : une oscillation d'une seule fréquence dans une cloche. Glissée le long de l'époque, elle donne à chaque
  instant la puissance de cette fréquence (et sa phase, 2.3). Plus de cycles : la fréquence est plus précise, le temps moins (le tableau
  sous 2.1).
- Rouge : **synchronisation** (la puissance monte) ; bleu : **désynchronisation** (elle baisse) (Pfurtscheller et Lopes da Silva, 1999).
- La puissance est calculée dans chaque essai puis moyennée : c'est la puissance **totale** (calée en phase sur le mot ou non ; 2.3).
- Chaque carte est en **dB par rapport à la même posture sans la tâche** (0 dB : comme sans la tâche) :

![référence](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_3_rythmes/reference_db.png)

- Près des bords de l'époque, l'ondelette déborde des données : ces zones restent blanches. La période avant le mot ne sert pas de
  référence : ce n'est pas un repos (la tape précédente, et les pas en marche).
- Le **test par amas** (notebook 2, 2.2) regroupe les points voisins de la carte qui dépassent un seuil (un **amas**) et les compare au
  hasard. Il **explore** la carte ; le test de votre hypothèse reste celui de la partie 4, sur une mesure choisie à l'avance.

### 2.1 Le temps-fréquence de chaque participant

La cellule calcule, pour chaque participant, posture et type de mot, la puissance de Morlet de 3 à 30 Hz toutes les 20 ms, et la
référence de chaque posture (le code : `ob.calculer_tf`). Environ 1 minute ; si vous ne changez que la bande, la fenêtre ou la région, le
calcul précédent sert encore.

In [ ]:
# ---- 2.1 Le temps-fréquence de chaque participant (le code : ob.calculer_tf ; environ 1 minute)
cle = (CENTRE, HYPOTHESE == 'tr', tuple(N_CYCLES))           # ce qui change les cartes
if globals().get('TF') is None or TF['cle'] != cle:
    TF = ob.calculer_tf(EPOQUES, HYPOTHESE, sans_tache, enregistrements, n_cycles=N_CYCLES)
    TF['cle'] = cle
differences, cartes_ab = ob.cartes_groupe(TF, CONDITIONS, COND_A, COND_B)   # par participant : B - A (dB) ; A et B (dB, sans la tâche)
participants = list(differences)

temps_tf, FREQUENCES = TF['temps'], ob.FREQUENCES_TF
dans_bande = (FREQUENCES >= BANDE[0]) & (FREQUENCES <= BANDE[1])          # votre bande, dans les cartes
dans_fenetre = (temps_tf >= FENETRE[0] - 1e-9) & (temps_tf <= FENETRE[1] + 1e-9)   # votre fenêtre
AFFICHAGE = (-0.5, 1.0) if CENTRE == 'mot' else (-1.0, 0.5)                # les instants montrés (s)
montres = (temps_tf >= AFFICHAGE[0] - 1e-9) & (temps_tf <= AFFICHAGE[1] + 1e-9)
info_region = mne.create_info(['région'], 100 / ob.DECIM_TF, 'misc')
tr_median = np.median(np.concatenate([EPOQUES[p].metadata['tr_ms'].to_numpy() for p in participants])) / 1000

resolution = ob.LARGEUR_CLOCHE * TF['sigma_t']              # la résolution en temps de chaque fréquence (s)
print(f'{len(participants)} participants pour la différence {COND_B} - {COND_A}, {len(cartes_ab)} pour les cartes de chaque condition')
print(f'Dans votre bande, la puissance à un instant résume {resolution[dans_bande].min() * 1000:.0f} à {resolution[dans_bande].max() * 1000:.0f} ms '
      f'de signal : votre fenêtre de {FENETRE[0] * 1000:.0f} à {FENETRE[1] * 1000:.0f} ms mesure en fait environ de '
      f'{(FENETRE[0] - resolution[dans_bande].max() / 2) * 1000:.0f} à {(FENETRE[1] + resolution[dans_bande].max() / 2) * 1000:.0f} ms')
pd.DataFrame({'cycles': N_CYCLES, 'résolution en temps (ms)': resolution * 1000, 'résolution en fréquence (Hz)': ob.LARGEUR_CLOCHE / (2 * np.pi * TF['sigma_t'])},
             index=pd.Index(FREQUENCES, name='Hz')).iloc[[1, 3, 5, 7, 9, 12, 17, 22, 27]].round(1).T

In [ ]:
# ---- F3 : les cartes temps-fréquence du groupe, dans votre région
A = np.array([ob.dans_la_region(a, j) for a, _ in cartes_ab.values()])   # participants x fréquences x instants (dB)
B = np.array([ob.dans_la_region(b, j) for _, b in cartes_ab.values()])
D = np.array([ob.dans_la_region(d, j) for d in differences.values()])
D = D[~np.isnan(D).any(axis=(1, 2))]                         # sans un participant dont aucune électrode de la région n'est mesurée
amas, p_amas, significatif, X = ob.amas_tf(D, TF['bord'], montres)   # le test par amas, sans les bords
moyennes = [np.where(TF['bord'], np.nan, np.nanmean(v, axis=0))[:, montres] for v in (A, B, D)]   # les bords en blanc
limite = np.nanmax(np.abs(np.concatenate(moyennes[:2])))     # la même échelle pour les deux conditions
ob.figure_tf(moyennes, [f'{COND_A} ({len(A)} participants)', f'{COND_B} ({len(B)} participants)', f'{COND_B} - {COND_A} ({len(D)} participants)'],
             [limite, limite, np.nanmax(np.abs(moyennes[2]))], [None, None, significatif], temps_tf[montres], FREQUENCES, info_region,
             len(D), FENETRE, BANDE, ZERO, tr_median, 'dB',
             f'F3 — {NOM}, région {", ".join(REGION)} : dB par rapport à la même posture sans la tâche ; tirets : '
             f'{"la tape" if CENTRE == "mot" else "le mot"} au TR médian ; cadre : votre mesure ; contour : amas à p < .05')
print(ob.decrire_amas(amas, p_amas, X, temps_tf[montres], COND_B, 'puissance'))

**Lire.** Quelles bandes changent après le zéro (le mot, ou la tape pour une mesure « (tape) »), et dans quel sens ? La différence est-elle
dans le cadre de votre mesure ?

### <span style="background:#7C3AED; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 3</span> À vous de construire : la carte d'un participant

Objectif : la carte temps-fréquence de la différence `COND_B − COND_A` de **P010**, dans votre région, avec votre cadre.

<div style="border-left:5px solid #7C3AED; background:#F5F0FF; color:#1F2937; padding:10px 16px; border-radius:6px; margin:8px 0;"><b style="color:#7C3AED;">Indices</b><br><ul style='margin:4px 0 0 0;'><li><code>differences['P010']</code> : électrodes × fréquences × instants (dB). <code>ob.dans_la_region(carte, j)</code> fait la moyenne de votre région.</li><li><code>plt.pcolormesh(temps_tf, FREQUENCES, carte, cmap='RdBu_r', vmin=-lim, vmax=lim, shading='auto')</code>, avec <code>lim = np.nanmax(np.abs(carte))</code>.</li><li>Le cadre : <code>plt.gca().add_patch(plt.Rectangle((FENETRE[0], BANDE[0]), FENETRE[1] - FENETRE[0], BANDE[1] - BANDE[0], fill=False))</code>.</li></ul></div>

In [ ]:
# ---- Votre cellule

In [ ]:
#@title Solution (à consulter après avoir essayé)
carte_p = ob.dans_la_region(differences["P010"], j)
lim = np.nanmax(np.abs(carte_p))
plt.figure(figsize=(7, 3.5))
plt.pcolormesh(temps_tf, FREQUENCES, carte_p, cmap="RdBu_r", vmin=-lim, vmax=lim, shading="auto")
plt.colorbar(label="dB")
plt.gca().add_patch(plt.Rectangle((FENETRE[0], BANDE[0]), FENETRE[1] - FENETRE[0], BANDE[1] - BANDE[0], fill=False, lw=1.5))
plt.axvline(0, color="k", lw=0.8)
plt.xlabel(f"Temps depuis {ZERO} (s)"); plt.ylabel("Fréquence (Hz)"); plt.title(f"P010 : {COND_B} - {COND_A}")
plt.show()
# Un seul participant est bien plus bruité que le groupe (F3) : c'est pourquoi on teste sur le groupe.

### 2.2 Les essais un par un

La puissance de votre bande, essai par essai, pour `PARTICIPANT` (cellule 0.4) : la condition A en bas, la condition B en haut, chacune du
TR le plus court au plus long. Le trait noir marque l'autre événement (la tape, ou le mot) : une hausse liée au zéro forme une bande
verticale ; une hausse liée à l'autre événement suit le trait.

In [ ]:
# ---- 2.2 La puissance de votre bande, essai par essai, chez un participant (le code : ob.figure_essais_tf)
ob.figure_essais_tf(TF, EPOQUES, CONDITIONS, COND_A, COND_B, PARTICIPANT, j, dans_bande, CENTRE, ZERO, AFFICHAGE, BANDE)

### 2.3 La part calée sur l'événement : la cohérence de phase

La puissance totale (2.1) ne dit pas si l'oscillation a la même phase d'un essai à l'autre. La **cohérence de phase** le mesure, à chaque
instant et chaque fréquence : proche de 1, la phase est la même dans tous les essais (l'oscillation est calée sur le mot et se voit dans
l'ERP) ; proche de 0, elle est au hasard.

![phase](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_3_rythmes/phase.png)

La **PPC** (cohérence de phase par paires ; Vinck et al., 2010) compare la phase de chaque paire d'essais. Elle estime le carré de la
cohérence de phase sans le biais du nombre d'essais (avec peu d'essais, des phases au hasard semblent déjà alignées) : deux conditions
qui n'ont pas le même nombre d'essais se comparent directement. En marche, le bruit des pas (pas calé sur le mot) la baisse aussi.

In [ ]:
# ---- 2.3 La cohérence de phase entre essais, dans votre région (le code : ob.coherences_groupe)
coherences = ob.coherences_groupe(TF, CONDITIONS, COND_A, COND_B)   # par participant : la PPC de A et de B
A_c = np.array([ob.dans_la_region(a, j) for a, _ in coherences.values()])   # participants x fréquences x instants
B_c = np.array([ob.dans_la_region(b, j) for _, b in coherences.values()])
D_c = np.array([ob.dans_la_region(b - a, j) for a, b in coherences.values()])
garde_c = ~np.isnan(D_c).any(axis=(1, 2))                     # sans un participant dont aucune électrode de la région n'est mesurée
A_c, B_c, D_c = A_c[garde_c], B_c[garde_c], D_c[garde_c]
amas_c, p_amas_c, significatif_c, X_c = ob.amas_tf(D_c, TF['bord'], montres)
moyennes_c = [np.where(TF['bord'], np.nan, v.mean(axis=0))[:, montres] for v in (A_c, B_c, D_c)]
limite_c = np.nanmax(np.abs(np.concatenate(moyennes_c[:2])))
ob.figure_tf(moyennes_c, [f'{COND_A} ({len(A_c)} participants)', f'{COND_B} ({len(B_c)} participants)', f'{COND_B} - {COND_A} ({len(D_c)} participants)'],
             [limite_c, limite_c, np.nanmax(np.abs(moyennes_c[2]))], [None, None, significatif_c], temps_tf[montres], FREQUENCES,
             info_region, len(D_c), FENETRE, BANDE, ZERO, tr_median, 'PPC',
             f'La cohérence de phase entre essais (PPC), région {", ".join(REGION)} : 0 = une phase au hasard ; cadre : votre mesure ; '
             f'contour : amas à p < .05')
cadre_c = [v[:, dans_bande][:, :, dans_fenetre].mean(axis=(1, 2)) for v in (A_c, B_c, D_c)]   # chaque participant, dans votre cadre
t, p = stats.ttest_1samp(cadre_c[2], 0.0)
print(f'PPC dans votre cadre : {COND_A} {cadre_c[0].mean():.3g}, {COND_B} {cadre_c[1].mean():.3g} ; {COND_B} - {COND_A} '
      f'{cadre_c[2].mean():+.3g}, t({len(cadre_c[2]) - 1}) = {t:.2f}, p = {p:.3f}, dz = {cadre_c[2].mean() / cadre_c[2].std(ddof=1):.2f}')
print(ob.decrire_amas(amas_c, p_amas_c, X_c, temps_tf[montres], COND_B, 'cohérence'))

### <span style="background:#5B6472; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Pour aller plus loin</span> <span style="background:#E8710A; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 2</span> 2.4 Le compromis entre le temps et la fréquence

Une ondelette avec **plus de cycles** mesure la fréquence plus précisément, mais elle s'étale plus dans le temps : un effet court se
dilue. Avec **moins de cycles**, c'est l'inverse (Cohen, 2019). Le tableau sous 2.1 donne, pour chaque fréquence, les deux résolutions.

**À faire.** Dans 0.4, mettez `N_CYCLES = 7 * np.ones(len(ob.FREQUENCES_TF))` (7 cycles à toutes les fréquences), relancez à partir de
0.5 (environ 1 minute), puis regardez F3, le tableau des résolutions et votre mesure (partie 4). Refaites avec 3 cycles. Votre effet
change-t-il ? Dans quel sens, et pourquoi ?

## 3. Les topographies

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="color:#0B5CAD; font-weight:bold;">toutes les hypothèses</span></div>

Votre bande dans votre fenêtre, électrode par électrode, vue de dessus (nez en haut), en dB par rapport à la même posture sans la tâche.
Points noirs : votre région ; cercles vides : électrodes mesurées chez trop peu de participants (moins de 3, ou de la moitié) ; ronds
blancs sur la différence : p < .05 à cette électrode (sans correction).

In [ ]:
# ---- F4 : les topographies
valeurs_A = np.array([ob.dans_le_cadre(a, dans_bande, dans_fenetre) for a, _ in cartes_ab.values()])   # participants x électrodes
valeurs_B = np.array([ob.dans_le_cadre(b, dans_bande, dans_fenetre) for _, b in cartes_ab.values()])
valeurs_D = np.array([ob.dans_le_cadre(d, dans_bande, dans_fenetre) for d in differences.values()])
vue = np.isfinite(valeurs_D).sum(axis=0) >= max(3, len(valeurs_D) / 2)   # mesurée chez 3 participants et la moitié d'entre eux au moins
visibles = np.flatnonzero(vue)
cartes = [np.nanmean(v[:, visibles], axis=0) for v in (valeurs_A, valeurs_B, valeurs_D)]
p_electrodes = np.array([stats.ttest_1samp(valeurs_D[np.isfinite(valeurs_D[:, k]), k], 0.0).pvalue for k in visibles])
info_vue = mne.pick_info(EPOQUES[participants[0]].info, visibles)
noms = np.array(noms_ch)
ob.figure_topos_tf(cartes, info_vue, EPOQUES[participants[0]].info, vue, noms, REGION, p_electrodes,
                   [COND_A, COND_B, f'{COND_B} - {COND_A}'],
                   f'F4 — {NOM} : {BANDE[0]} à {BANDE[1]} Hz, de {FENETRE[0] * 1000:.0f} à {FENETRE[1] * 1000:.0f} ms depuis {ZERO}')
print('Électrodes à p < .05 (non corrigé) :', ', '.join(noms[vue][p_electrodes < 0.05]) or 'aucune')
print('Électrodes trop peu mesurées pour les cartes (cercles vides) :', ', '.join(noms[~vue]) or 'aucune')

## 4. Une mesure par participant : le test de votre hypothèse

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="color:#0B5CAD; font-weight:bold;">toutes les hypothèses</span></div>

La moyenne de la carte de chaque participant dans votre cadre (bande × fenêtre × région) ; test t et **dz** (1.2). C'est **le test de
votre rapport**. Chaque condition est aussi comparée à 0 dB (la même posture sans la tâche), et la différence est refaite en µV² (une
soustraction au lieu d'un rapport) : si la conclusion change, elle dépend de la façon de comparer deux puissances (Keil et al., 2022).

In [ ]:
# ---- 4.1 Votre mesure pour chaque participant (le code : ob.mesures_participants_tf)
mesures_tf = ob.mesures_participants_tf(TF, CONDITIONS, COND_A, COND_B, differences, cartes_ab, j, dans_bande, dans_fenetre)
mesures_tf.round(2)                                          # A et B (dB, sans la tâche), B - A (dB), et B - A (µV²)

In [ ]:
# ---- F5 : votre mesure dans les deux conditions ; le test de B contre A
deux = mesures_tf[[COND_A, COND_B]].dropna()
ob.figure_mesure_tf(mesures_tf, COND_A, COND_B, COULEURS, NOM)
for colonne, unite in ((f'{COND_B} - {COND_A}', 'dB'), (f'{COND_B} - {COND_A} (µV²)', 'µV²')):
    d = mesures_tf[colonne].dropna()
    t, p = stats.ttest_1samp(d, 0.0)
    print(f'{NOM}, {COND_B} - {COND_A} : {d.mean():+.2f} {unite}, t({len(d) - 1}) = {t:.2f}, p = {p:.3f}, dz = {d.mean() / d.std(ddof=1):.2f}')
for condition in (COND_A, COND_B):
    t, p = stats.ttest_1samp(deux[condition], 0.0)
    print(f'{condition} par rapport à la même posture sans la tâche : {deux[condition].mean():+.2f} dB, t({len(deux) - 1}) = {t:.2f}, p = {p:.3f}')

## 5. L'interaction avec la posture

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H4</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H8</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H10</span></div>

Comme au notebook 2 : l'effet de fréquence (LF − HF) dans chaque posture (ici, le rapport des puissances des deux types de mots, en dB,
dans votre cadre), le même effet sur le comportement, puis le test direct de l'**interaction** : l'effet en marche moins l'effet assis.
Pour H10, c'est le second volet : l'effet de fréquence sur le **bêta avant la tape**.

Les deux réglages de la cellule : `EFFET` (`'frequence'`, LF − HF ; ou `'lexicalite'`, pseudo-mots − mots, pour explorer) et
`COMPORTEMENT` (`'TR (ms)'`, `'Exactitude (%)'` ou `'IES (ms)'`).

In [ ]:
# ---- 5.1 L'effet dans chaque posture (le code : ob.effet_par_posture_tf)
EFFET = 'frequence'            # 'frequence' (LF - HF) ou 'lexicalite' (pseudo-mots - mots)
COMPORTEMENT = 'TR (ms)'       # 'TR (ms)', 'Exactitude (%)' ou 'IES (ms)'

NOM_EFFET = 'LF - HF' if EFFET == 'frequence' else 'pseudo-mots - mots'
interaction = ob.effet_par_posture_tf(TF, EFFET, j, dans_bande, dans_fenetre)       # EEG : un participant par ligne (dB)
interaction_c = ob.effet_comportement_par_posture(essais, EFFET, COMPORTEMENT)      # le comportement
print(f'{NOM} : effet {NOM_EFFET} (dB) dans chaque posture')
interaction.round(2)

In [ ]:
# ---- F6 : l'effet dans chaque posture, et l'interaction (marche - assis)
UNITE_C = COMPORTEMENT.split('(')[1].rstrip(')')             # 'ms' ou '%'
ob.figure_effets_posture([(interaction, NOM, 'dB'), (interaction_c, COMPORTEMENT, UNITE_C)], NOM_EFFET)
for tableau, nom, unite in ((interaction, NOM, 'dB'), (interaction_c, COMPORTEMENT, UNITE_C)):
    n = len(tableau)
    for posture in ('assis', 'marche'):                      # l'effet dans chaque posture, contre 0
        t, p = stats.ttest_1samp(tableau[posture], 0.0)
        print(f'{nom} : effet {posture} {tableau[posture].mean():+.2f} {unite}, t({n - 1}) = {t:.2f}, p = {p:.3f}')
    ecart = tableau['marche'] - tableau['assis']             # l'interaction : l'effet change-t-il avec la posture ?
    t, p = stats.ttest_rel(tableau['marche'], tableau['assis'])
    print(f'{nom} : interaction (marche - assis) {ecart.mean():+.2f} {unite}, t({n - 1}) = {t:.2f}, p = {p:.3f}, '
          f'dz = {ecart.mean() / ecart.std(ddof=1):.2f}\n')

## 6. L'EEG et le comportement

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H5</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H9</span></div>

Un point par participant : son effet B − A sur le comportement et sur votre mesure (partie 4), ou ses valeurs dans une condition ; la
droite et le r de Pearson. Exploratoire : 17 participants au plus. Pour H9, l'écart de TR entre lents et rapides existe par
construction : la question est de savoir si les participants dont l'écart est grand ont aussi une plus grande différence de bêta.

Deux réglages : `COMPORTEMENT` (avec H9, le TR seulement) et `VALEURS` (`'difference'` : l'effet B − A de chaque participant ; ou une
condition, par exemple `COND_A`).

In [ ]:
# ---- 6.1 Le comportement et la puissance de chaque participant
COMPORTEMENT = 'TR (ms)'       # 'TR (ms)', 'Exactitude (%)' ou 'IES (ms)'
VALEURS = 'difference'         # 'difference' ou une condition, par exemple COND_A

comport = ob.comportement(essais, HYPOTHESE, POSTURE)       # comme au notebook 2 (partie 1)
large_c = comport.pivot(index='participant', columns='condition', values=COMPORTEMENT)
if VALEURS == 'difference':
    x, y = large_c[COND_B] - large_c[COND_A], mesures_tf[f'{COND_B} - {COND_A}']   # l'effet de chaque participant
else:
    x, y = large_c[VALEURS], mesures_tf[VALEURS]                                 # les valeurs d'une condition
points = pd.DataFrame({'comportement': x, 'eeg': y}).dropna()                    # les participants qui ont les deux
points.round(2)

In [ ]:
# ---- F7 : le nuage de points
r, p = stats.pearsonr(points['comportement'], points['eeg'])
suffixe = f' ({COND_B} - {COND_A})' if VALEURS == 'difference' else f' ({VALEURS})'
ob.figure_nuage(points, COMPORTEMENT + suffixe, f'{NOM} (dB)' + suffixe)
print(f'Pearson : r = {r:.2f}, p = {p:.3f} (n = {len(points)})')

### <span style="background:#E8710A; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 2</span> À vous de compléter : un lien robuste ?

Comme au notebook 2 : un seul participant extrême peut faire (ou défaire) un r de Pearson. Calculez le **rho de Spearman**
(`stats.spearmanr`), puis refaites Pearson sans le participant le plus éloigné.

In [ ]:
# ---- Exercice
# rho, p_rho = stats.spearmanr(points['comportement'], ...)                     # <--- à compléter
# extreme = ((points - points.mean()) / points.std()).abs().max(axis=1).idxmax()  # le participant le plus éloigné
# r_sans, p_sans = stats.pearsonr(*points.drop(index=...).T.to_numpy())         # <--- à compléter

In [ ]:
#@title Solution (à consulter après avoir essayé)
rho, p_rho = stats.spearmanr(points['comportement'], points['eeg'])
extreme = ((points - points.mean()) / points.std()).abs().max(axis=1).idxmax()
r_sans, p_sans = stats.pearsonr(*points.drop(index=extreme).T.to_numpy())
print(f'Spearman : rho = {rho:.2f}, p = {p_rho:.3f} ; Pearson sans {extreme} : r = {r_sans:.2f}, p = {p_sans:.3f}')
# Si le lien disparaît sans un seul participant, il ne faut pas le présenter comme un résultat.

## 7. À retenir pour votre rapport

| Votre question | Hypothèses | Le test principal | Les figures |
|---|---|---|---|
| A et B diffèrent-ils ? | H1, H2, H3, H6, H7, H9, H10 | le test t sur la puissance de votre mesure (partie 4) | le temps-fréquence (2), la carte (3) |
| L'effet change-t-il en marche ? | H4, H8, H10 | le test t apparié sur l'effet, marche contre assis (partie 5) | l'effet dans chaque posture (5) |
| L'EEG suit-il le comportement ? | H5, H9 | le r de Pearson (partie 6) | le nuage de points (6) |

<div style="border-left:5px solid #15803D; background:#ECF8EF; color:#1F2937; padding:10px 16px; border-radius:6px; margin:8px 0;"><b style="color:#15803D;">À retenir</b><br><ul style='margin:4px 0 0 0;'><li>Énoncez l'hypothèse et vos prédictions (bande, fenêtre, région, sens de l'effet) <b>avant</b> les résultats.</li><li>Pour chaque test : la moyenne de chaque condition, t(ddl), p, dz et le nombre de participants.</li><li>Pour la posture : rapportez aussi la comparaison avec et sans la tâche (1.2), la pente et la bosse (1.3) et le résultat en µV² (4).</li><li>Un effet du thêta : regardez sa cohérence de phase (2.3) ; la part calée sur le mot appartient à l'ERP du notebook 2.</li><li>Un résultat nul est un résultat : rapportez-le avec la même précision.</li></ul></div>

**Pistes pour aller plus loin** : votre effet tient-il quand on change un choix raisonnable ? Refaites les parties 0.5 à 4 en changeant
**une** option à la fois (0.4), et notez t, p et dz dans un tableau :

| Changement | Pourquoi |
|---|---|
| `POSTURE = 'assis'`, puis `'marche'` | l'effet existe-t-il dans chaque posture ? |
| la ligne de l'alpha (posture) | le thêta porte le bruit des pas |
| `N_CYCLES` (2.4) | le compromis entre le temps et la fréquence |
| la différence en µV² plutôt qu'en dB (4) | la façon de comparer deux puissances |

**Limites à discuter** : 17 participants ; les tests non corrigés (électrode par électrode) ; la résolution des ondelettes (une fenêtre de
thêta mesure aussi les 150 ms qui l'entourent) ; le bruit de la marche, surtout dans le thêta ; une marche qui change avec la tâche, et la
marche seule toujours avant les blocs en marche ; le côté de la tape pour la lexicalité ; les deux postures nettoyées par des chaînes un
peu différentes (notebook 1). Pour H9 : avant la tape, le mot n'est pas au même moment dans les essais rapides et lents.

## 8. Défis (points bonus)

Pour les groupes qui veulent aller plus loin : cinq défis, les mêmes questions qu'au notebook 2, pour les rythmes, avec les **outils de ce
notebook** (une option de 0.4, une cellule à relancer, un tableau à relire). Il n'y a pas de solution : c'est votre travail. Pour chacun,
présentez dans les **annexes du rapport** :

1. la question, et pourquoi elle compte pour votre hypothèse ;
2. ce que vous avez changé (la mesure, la posture, la région) ;
3. le résultat (t, p, dz, ou une figure), et votre interprétation en deux ou trois phrases.

**Une règle** : remettez les options à leur valeur de départ (0.4) après chaque défi, et relancez à partir de 0.5.

### <span style="background:#B45309; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Défi</span> A. Le thêta tardif, aligné sur le mot ou sur la réponse ?

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H6</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H7</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H8</span></div>

**La question.** Le défi A du notebook 2, pour les rythmes. Les mots rares sont reconnus plus lentement (environ 75 ms) : si le thêta
tardif suit la **réponse** plutôt que le mot, une différence LF − HF peut venir du seul moment de la réponse (2.2 : les essais un par un).
On mesure donc le même thêta **juste avant la tape**.

**Comment.**

1. Dans 0.4, ajoutez la ligne `MESURE, FENETRE = 'thêta (tape)', (-0.3, 0.0)` : les époques de la tape, les 300 ms avant la réponse
   (la bande et la région ne changent pas).
2. Relancez à partir de 0.5 jusqu'à la partie 4 (environ 1 minute).

**À rapporter.** L'effet (t, p, dz) aligné sur le mot (votre résultat) et aligné sur la réponse. Lequel est le plus net ? Que dit-il de
votre hypothèse ?

In [ ]:
# ---- Défi A : votre cellule (gardez-y les résultats que vous rapportez)

### <span style="background:#B45309; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Défi</span> B. Un contrôle positif : des effets connus des rythmes

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="color:#0B5CAD; font-weight:bold;">toutes les hypothèses</span></div>

**La question.** Le défi B du notebook 2, pour les rythmes. Deux effets sont bien établis : l'**alpha baisse** pendant la lecture, par
rapport au repos (Pfurtscheller et Lopes da Silva, 1999), et le **bêta baisse** avant un mouvement, la tape (Steinemann et al., 2018).
Si le notebook les trouve, un résultat nul pour votre hypothèse est plus crédible.

**Comment.**

1. Dans 0.4, ajoutez la ligne `MESURE, BANDE, FENETRE, REGION = 'alpha', (8, 12), (0.3, 0.8), PO` ; relancez jusqu'à la partie 4.
   Regardez les lignes « par rapport à la même posture sans la tâche » de F5 (et 1.2 : « tâche assis − repos »).
2. Même chose avec `MESURE, BANDE, FENETRE, REGION = 'bêta (tape)', (13, 30), (-0.5, 0.0), C`.

**À rapporter.** Les deux effets connus (dB, t, p) : le notebook les trouve-t-il, dans les deux postures ? Que dit-il de la sensibilité
de l'analyse, et donc de votre résultat ?

In [ ]:
# ---- Défi B : votre cellule (gardez-y les résultats que vous rapportez)

### <span style="background:#B45309; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Défi</span> C. Une fenêtre choisie sans tricher

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H2</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H3</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H7</span></div>

**La question.** Le défi C du notebook 2, pour les rythmes : choisir la fenêtre là où la différence est la plus forte, puis la tester sur
les mêmes données, rend le test trop optimiste. On choisit l'instant sur les essais **assis**, et on le teste sur les essais **en marche**.

**Comment.**

1. Dans 0.4, `POSTURE = 'assis'` ; relancez jusqu'à F3. L'instant où la différence moyenne, dans votre bande et votre région, est la plus
   loin de 0 (entre 0,1 et 0,8 s) :
   `profil = np.nanmean(D[:, dans_bande], axis=(0, 1))`, `dedans = (temps_tf >= 0.1) & (temps_tf <= 0.8)`, puis
   `pic = temps_tf[dedans][np.argmax(np.abs(profil[dedans]))]` (`D` : la cellule F3).
2. Dans 0.4, `POSTURE = 'marche'` et `FENETRE = (pic - 0.1, pic + 0.1)` ; relancez jusqu'à la partie 4.

**À rapporter.** La fenêtre choisie (assis), et l'effet dans cette fenêtre en marche (t, p, dz). L'effet se retrouve-t-il dans des
données qui n'ont pas servi à le choisir ?

In [ ]:
# ---- Défi C : votre cellule (gardez-y les résultats que vous rapportez)

### <span style="background:#B45309; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Défi</span> D. À gauche ou à droite ?

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="color:#0B5CAD; font-weight:bold;">toutes les hypothèses</span></div>

**La question.** Le défi D du notebook 2, pour les rythmes : l'effet est-il plus net au-dessus d'un hémisphère ?

**Comment.**

1. Dans 0.4, `REGION = ['C3', 'P3']` (pour le thêta précoce : `['F3', 'C3']` ; pour le bêta : `['C3']`) ; relancez à partir de 0.5 jusqu'à
   la partie 4 (le temps-fréquence n'est pas recalculé), et gardez l'effet de chaque participant :
   `gauche = mesures_tf[f'{COND_B} - {COND_A}']` (cellule 4.1).
2. Même chose de l'autre côté (`['C4', 'P4']`, `['F4', 'C4']` ou `['C4']`) : `droite = ...`.
3. Comparez : `stats.ttest_rel(gauche, droite, nan_policy='omit')`.

**À rapporter.** L'effet à gauche, à droite, et leur différence (t, p). Pour le bêta avant la tape (H9, H10) : la main qui répond fait
baisser le bêta au-dessus de l'hémisphère opposé (C3 pour la main droite) ; tenez-en compte.

In [ ]:
# ---- Défi D : votre cellule (gardez-y les résultats que vous rapportez)

### <span style="background:#B45309; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Défi</span> E. Un rythme, ou la pente ?

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H1</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H6</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H10</span></div>

**La question.** La marche élève la puissance à toutes les fréquences (le bruit des pas) : une différence en dB peut venir de la **pente**
du spectre plutôt que d'un **rythme** (1.3 ; Gyurkovics et al., 2021). Votre effet de posture est-il un changement du rythme ?

**Comment.** Relisez les tableaux de 1.2 et de 1.3 pour votre bande et votre région : la différence « tâche en marche − marche seule »
et « marche seule − repos » en dB (1.2), puis pour la pente et pour la bosse (« au-dessus de la pente », 1.3). Refaites-le avec la ligne
de l'alpha (0.4).

**À rapporter.** Pour votre bande : la marche change-t-elle surtout la pente, ou la bosse ? Que reste-t-il de votre effet de posture une
fois la pente mise de côté ? Et dans l'alpha ?

In [ ]:
# ---- Défi E : votre cellule (gardez-y les résultats que vous rapportez)

**Suite** : `brainwalk_4_apprentissage.ipynb` (décoder).

<details><summary>Références</summary>

- Babiloni, C., Barry, R. J., Başar, E., Blinowska, K. J., Cichocki, A., Drinkenburg, W. H. I. M., Klimesch, W., Knight, R. T., Lopes da Silva, F., Nunez, P., Oostenveld, R., Jeong, J., Pascual-Marqui, R., Valdes-Sosa, P., & Hallett, M. (2020). International Federation of Clinical Neurophysiology (IFCN) – EEG research workgroup: Recommendations on frequency and topographic analysis of resting state EEG rhythms. Part 1: Applications in clinical research studies. *Clinical Neurophysiology, 131*(1), 285-307. https://doi.org/10.1016/j.clinph.2019.06.234
- Bastiaansen, M. C. M., van der Linden, M., ter Keurs, M., Dijkstra, T., & Hagoort, P. (2005). Theta responses are involved in lexical-semantic retrieval during language processing. *Journal of Cognitive Neuroscience, 17*(3), 530-541. https://doi.org/10.1162/0898929053279469
- Cohen, M. X. (2019). A better way to define and describe Morlet wavelets for time-frequency analysis. *NeuroImage, 199*, 81-86. https://doi.org/10.1016/j.neuroimage.2019.05.048
- Donner, T. H., Siegel, M., Fries, P., & Engel, A. K. (2009). Buildup of choice-predictive activity in human motor cortex during perceptual decision making. *Current Biology, 19*(18), 1581-1585. https://doi.org/10.1016/j.cub.2009.07.066
- Donoghue, T., Haller, M., Peterson, E. J., Varma, P., Sebastian, P., Gao, R., Noto, T., Lara, A. H., Wallis, J. D., Knight, R. T., Shestyuk, A., & Voytek, B. (2020). Parameterizing neural power spectra into periodic and aperiodic components. *Nature Neuroscience, 23*(12), 1655-1665. https://doi.org/10.1038/s41593-020-00744-x
- Grabner, R. H., Brunner, C., Leeb, R., Neuper, C., & Pfurtscheller, G. (2007). Event-related EEG theta and alpha band oscillatory responses during language translation. *Brain Research Bulletin, 72*(1), 57-65. https://doi.org/10.1016/j.brainresbull.2007.01.001
- Gwin, J. T., Gramann, K., Makeig, S., & Ferris, D. P. (2010). Removal of movement artifact from high-density EEG recorded during walking and running. *Journal of Neurophysiology, 103*(6), 3526-3534. https://doi.org/10.1152/jn.00105.2010
- Gyurkovics, M., Clements, G. M., Low, K. A., Fabiani, M., & Gratton, G. (2021). The impact of 1/f activity and baseline correction on the results and interpretation of time-frequency analyses of EEG/MEG data: A cautionary tale. *NeuroImage, 237*, 118192. https://doi.org/10.1016/j.neuroimage.2021.118192
- Harper, J., Malone, S. M., & Bernat, E. M. (2014). Theta and delta band activity explain N2 and P3 ERP component activity in a go/no-go task. *Clinical Neurophysiology, 125*(1), 124-132. https://doi.org/10.1016/j.clinph.2013.06.025
- Jurkiewicz, M. T., Gaetz, W. C., Bostan, A. C., & Cheyne, D. (2006). Post-movement beta rebound is generated in motor cortex: Evidence from neuromagnetic recordings. *NeuroImage, 32*(3), 1281-1289. https://doi.org/10.1016/j.neuroimage.2006.06.005
- Keil, A., Bernat, E. M., Cohen, M. X., Ding, M., Fabiani, M., Gratton, G., Kappenman, E. S., Maris, E., Mathewson, K. E., Ward, R. T., & Weisz, N. (2022). Recommendations and publication guidelines for studies using frequency domain and time-frequency domain analyses of neural time series. *Psychophysiology, 59*(5), e14052. https://doi.org/10.1111/psyp.14052
- Klimesch, W., Schack, B., Schabus, M., Doppelmayr, M., Gruber, W., & Sauseng, P. (2004). Phase-locked alpha and theta oscillations generate the P1-N1 complex and are related to memory performance. *Cognitive Brain Research, 19*(3), 302-316. https://doi.org/10.1016/j.cogbrainres.2003.11.016
- Malcolm, B. R., Foxe, J. J., Butler, J. S., Molholm, S., & De Sanctis, P. (2018). Cognitive load reduces the effects of optic flow on gait and electrocortical dynamics during treadmill walking. *Journal of Neurophysiology, 120*(5), 2246-2259. https://doi.org/10.1152/jn.00079.2018
- Maris, E., & Oostenveld, R. (2007). Nonparametric statistical testing of EEG- and MEG-data. *Journal of Neuroscience Methods, 164*(1), 177-190. https://doi.org/10.1016/j.jneumeth.2007.03.024
- Peskar, M., Omejc, N., Šömen, M. M., Miladinović, A., Gramann, K., & Marusic, U. (2023). Stroop in motion: Neurodynamic modulation underlying interference control while sitting, standing, and walking. *Biological Psychology, 178*, 108543. https://doi.org/10.1016/j.biopsycho.2023.108543
- Pfurtscheller, G., & Lopes da Silva, F. H. (1999). Event-related EEG/MEG synchronization and desynchronization: Basic principles. *Clinical Neurophysiology, 110*(11), 1842-1857. https://doi.org/10.1016/S1388-2457(99)00141-8
- Protzak, J., & Gramann, K. (2021). EEG beta-modulations reflect age-specific motor resource allocation during dual-task walking. *Scientific Reports, 11*, 16110. https://doi.org/10.1038/s41598-021-94874-2
- Roach, B. J., & Mathalon, D. H. (2008). Event-related EEG time-frequency analysis: An overview of measures and an analysis of early gamma band phase locking in schizophrenia. *Schizophrenia Bulletin, 34*(5), 907-926. https://doi.org/10.1093/schbul/sbn093
- Sassenhagen, J., & Draschkow, D. (2019). Cluster-based permutation tests of MEG/EEG data do not establish significance of effect latency or location. *Psychophysiology, 56*(6), e13335. https://doi.org/10.1111/psyp.13335
- Scaltritti, M., Job, R., Alario, F.-X., & Sulpizio, S. (2020). On the boundaries between decision and action: Effector-selective lateralization of beta-frequency power is modulated by the lexical frequency of printed words. *Journal of Cognitive Neuroscience, 32*(11), 2131-2144. https://doi.org/10.1162/jocn_a_01606
- Steinemann, N. A., O'Connell, R. G., & Kelly, S. P. (2018). Decisions are expedited through multiple neural adjustments spanning the sensorimotor hierarchy. *Nature Communications, 9*, 3627. https://doi.org/10.1038/s41467-018-06117-0
- Vinck, M., van Wingerden, M., Womelsdorf, T., Fries, P., & Pennartz, C. M. A. (2010). The pairwise phase consistency: A bias-free measure of rhythmic neuronal synchronization. *NeuroImage, 51*(1), 112-122. https://doi.org/10.1016/j.neuroimage.2010.01.073
- Yordanova, J., & Kolev, V. (1998). Single-sweep analysis of the theta frequency band during an auditory oddball task. *Psychophysiology, 35*(1), 116-126. https://doi.org/10.1111/1469-8986.3510116
</details>